# CCCOH: GENIE covariance matrices

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
syst_name = ("GENIE",)
output_str = "genie"

In [3]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot

import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
from analysis_village.unfolding.covariance import *
from analysis_village.unfolding.utils import *

from cohpi_topologies import *
from var_config_cohpi import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

## 1. Open MC df

### 1 - 1: File path and name

In [4]:
input_path = "/data/sungbino/sbnd/v10_14_02/cohpi/"

## -- this file is based on 
####    geniewgtdf = geniesyst.geniesyst(f, nudf.ind, multisim_nuniv=100, slim=True, systematics=None)
####    bnbwgtdf = bnbsyst.bnbsyst(f, nudf.ind, multisim_nuniv=100, slim=True)
mc_file_name = "mc_MCP2025C_FallValidationII_prodgenie_corsika_proton_rockbox0p1_sbnd_CV_caf_flat_caf_sbnd_cohpi_wienersvd_slim_genie_clip_wgt.df"

mc_file_path = path.join(input_path, mc_file_name)

### 1 - 2: Check df keys and n_split

In [5]:
print("n split", get_n_split(mc_file_path))
print("Keys in mc file:")
print_keys(mc_file_path)

n split 2
Keys in mc file:
Keys: ['/evt_0', '/evt_1', '/hdr_0', '/hdr_1', '/histgenevtdf_0', '/histgenevtdf_1', '/histpotdf_0', '/histpotdf_1', '/mcnu_0', '/mcnu_1', '/pot_0', '/pot_1', '/split']


### 1 - 3: Load the dfs

Note:
- mc_evt_df is skimmed df with a single level for columns.
- mc_mcnu_df is skimmed df but with two levels for columns. second level is for multiverse weights

In [6]:
mc_keys2load = ['hdr', "mcnu", 'evt']
mc_dfs = load_dfs(mc_file_path, mc_keys2load, n_max_concat=10)
mc_evt_df = mc_dfs['evt']
mc_hdr_df = mc_dfs['hdr']
mc_mcnu_df = mc_dfs['mcnu']

In [13]:
mc_evt_df

,,,is_fv,is_not_clear_cosmic,n_prong,n_trk,n_proton,n_shower,tmatch_idx,tmatch_eff,tmatch_purity,long_dirx,...,is_muon_contained,is_cpi_contained,range_p_mu,range_p_cpi,total_pe,sum_integ_4cm,sum_integ_3cm,sum_integ_2cm,sum_integ_1cm,sum_integ_ratio_3over4sub
__ntuple,entry,rec.slc..index,,,,,,,,,,,,,,,,,,,,,
3,880,0,True,True,2,2,0,0,1,0.940908,0.986952,0.171236,...,False,True,0.691542,0.211966,14981.989258,9128.771484,7168.856445,4057.923584,-1.000000,3.657738
8,358,0,True,True,2,2,0,0,2,0.994158,0.798253,0.241840,...,False,False,0.275761,0.272023,5397.054688,3642.079590,1217.687012,-2.000000,-1.000000,0.502265
1,393,1,True,True,2,2,0,0,0,0.943338,0.999999,-0.116378,...,False,True,0.311163,0.113447,9529.189453,9722.078125,7135.074219,4846.637695,2421.743652,2.758045
4,701,0,True,True,2,2,0,0,0,0.853245,0.879007,-0.427309,...,False,True,0.440595,0.356263,30666.703125,6649.269043,5023.889160,2998.280029,1423.510986,3.090902
17,628,0,True,True,2,2,0,0,0,0.878665,0.951889,0.060349,...,False,False,0.510503,0.189539,24859.585938,8738.757812,5623.989746,3519.078613,278.585663,1.805589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,1136,0,True,True,2,2,0,0,1,0.931479,0.999765,-0.154638,...,False,True,0.879392,0.214776,12637.335938,9061.783203,6603.296875,3675.968750,-1.000000,2.685920
999,828,0,True,True,2,2,0,0,0,0.979789,0.914363,-0.065002,...,False,True,0.960049,0.223655,31425.421875,6968.430664,5026.367188,3132.542969,867.147827,2.588158
993,116,2,True,True,2,2,0,0,0,0.993179,0.901972,0.341063,...,False,True,0.198172,0.189006,13416.749023,8500.001953,6328.261230,4122.416016,-1.000000,2.913912


In [14]:
mc_mcnu_df

genie_mode true_E_nu true_p_mu true_p_pi  \
                                                                           
__ntuple entry rec.mc.nu..index                                            
0        0     0                         0  0.469397  0.366547       NaN   
         1     0                         1  0.427076       NaN       NaN   
               1                        10  1.086859  0.781676       NaN   
         2     0                        10  1.046138  0.573556       NaN   
         3     0                         0  0.391237  0.222466       NaN   
...                                    ...       ...       ...       ...   
976      1363  0                        10  0.641565       NaN       NaN   
         1364  0                        10  0.661321       NaN       NaN   
               1                         1  1.015625       NaN       NaN   
         1365  0                         1  1.303131  0.099012       NaN   
               1                         0  0.920195  0.482259       NaN   

                                true_cos_theta_mu true_cos_theta_pi    true_t  \
                                                                                
__ntuple entry rec.mc.nu..index                                                 
0        0     0                         0.850125               NaN  999999.0   
         1     0                              NaN               NaN  999999.0   
               1                         0.831460               NaN  999999.0   
         2     0                         0.668962               NaN  999999.0   
         3     0                         0.516203               NaN  999999.0   
...                                           ...               ...       ...   
976      1363  0                              NaN               NaN  999999.0   
         1364  0                              NaN               NaN  999999.0   
               1                              NaN               NaN  999999.0   
         1365  0                         0.869391               NaN  999999.0   
               1                         0.333604               NaN  999999.0   

                                nuint_categ     GENIE            ...  \
                                               univ_0    univ_1  ...   
__ntuple entry rec.mc.nu..index                                  ...   
0        0     0                          3  1.070001  2.652737  ...   
         1     0                         -1  0.344026  0.445954  ...   
               1                         -1  0.000000  0.000000  ...   
         2     0                         -1  0.000000  0.000000  ...   
         3     0                          3  0.999409  0.789339  ...   
...                                     ...       ...       ...  ...   
976      1363  0                         -1  0.134219  0.000000  ...   
         1364  0                         -1  0.000000  0.000000  ...   
               1                          0  0.829213  0.590666  ...   
         1365  0                         -1  0.496951  0.226989  ...   
               1                         -1  1.380852  0.408188  ...   

                                                                         \
                                  univ_90   univ_91   univ_92   univ_93   
__ntuple entry rec.mc.nu..index                                           
0        0     0                 1.870523  0.000000  2.502238  1.749138   
         1     0                 0.573680  0.883003  1.748859  0.295149   
               1                 0.000000  0.000000  0.000000  1.077655   
         2     0                 0.000000  0.000000  0.000000  0.935982   
         3     0                 1.005580  0.813434  0.971644  1.131130   
...                                   ...       ...       ...       ...   
976      1363  0                 0.277045  0.000310  0.029102  1.412128   
         1364  0                 0.000000  0.000000  0.000000  1.353453  

## 2. Normalization Variables

POT

In [ ]:
mc_tot_pot = mc_hdr_df['pot'].sum()
print("Total MC POT:", mc_tot_pot)
mc_pot_scale = 1.
mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))

tot_pot = mc_tot_pot

Volume

In [ ]:
TPC_Vol = (180. * 2.) * (190. * 2) * (250. - 10.) + (180. * 2.) * (190. + 100.) * (450. - 250.) # cm^3

Flux

In [ ]:
fluxfile = "/data/sungbino/sbnd/flux/sbnd_original_flux.root"
flux = uproot.open(fluxfile)
numu_flux = flux["flux_sbnd_numu"].to_numpy()
anumu_flux = flux["flux_sbnd_anumu"].to_numpy()
numu_flux_vals = numu_flux[0]
anumu_flux_vals = anumu_flux[0]

INTEGRATED_FLUX = (numu_flux_vals.sum() + anumu_flux_vals.sum()) / (1e4  * 1e6) # to cm2 # to POT
print("Integrated flux (numu + anumu) over total MC POT: %.3e per (cm^2 POT)" % INTEGRATED_FLUX)

## 3. match mc_evt_df and mc_mcnu_df

In [ ]:
mc_evt_df = ph.multicol_merge(mc_evt_df.reset_index(), mc_mcnu_df.reset_index(),
                               left_on=[("__ntuple", ""), ("entry", ""), ("tmatch_idx", "")],
                               right_on=[("__ntuple", ""), ("entry", ""), ("rec.mc.nu..index", "")],
                               how="left") ## -- save all sllices
mc_evt_df.loc[mc_evt_df['nuint_categ'].isna(), ['nuint_categ']] = -2

In [ ]:
mc_evt_df.nuint_categ.value_counts()

## 4. Spiltting to signal region (SR) and sideband (SB)

- Signal region: reco |t| < 0.04 (GeV/c)^2
- Sideband: 0.06 (GeV/c)^2 < reco |t| < 0.16 (GeV/c)^2

In [ ]:
mc_evt_df_sr = mc_evt_df[mc_evt_df.reco_t < 0.04]
mc_evt_df_sb = mc_evt_df[(mc_evt_df.reco_t > 0.06) & (mc_evt_df.reco_t < 0.16)]

## 5. Cross Section Variable Configs

P_mu

In [ ]:
mc_evt_df

## 5. Signal histograms

"All Signal in MC" are the same in SR and SB

In [ ]:
sighist_res = signal_hists(evtdf=mc_evt_df_sr, nudf=mc_mcnu_df, var_config=varcfg_p_mu, save_fig=False, save_name=None, return_data=True)
signal_hists(evtdf=mc_evt_df_sb, nudf=mc_mcnu_df, var_config=varcfg_p_mu, save_fig=False, save_name=None)

### 5 - 1: Response Matrix

In [ ]:
sighist_res["nevts_allmc"]

In [ ]:
reco_vs_true = get_smear_matrix(sighist_res["var_sel_truth"], sighist_res["var_sel_reco"], bins_2d=[varcfg_p_mu.bins, varcfg_p_mu.bins], plot=True)
eff = get_eff(reco_vs_true, sighist_res["nevts_allmc"])
response = get_response_matrix(reco_vs_true, eff, varcfg_p_mu.bins, plot=True)


## 6. Collect Histograms in CV and Multiverses

### 6 - 1: collect events in CV + multiverses, in each region

In [ ]:
cov_type = "xsec"
sr_signal_univ_events, sr_signal_sel_reco_cv, sr_bkg_univ_events, sr_bkg_sec_rec_cv = get_genie_univs(cov_type, mc_evt_df_sr, mc_mcnu_df, varcfg_p_mu, syst_name, flux=INTEGRATED_FLUX, pot=tot_pot, vol=TPC_Vol, topology_list=mode_list)
sb_signal_univ_events, sb_signal_sel_reco_cv, sb_bkg_univ_events, sb_bkg_sec_rec_cv = get_genie_univs(cov_type, mc_evt_df_sb, mc_mcnu_df, varcfg_p_mu, syst_name, flux=INTEGRATED_FLUX, pot=tot_pot, vol=TPC_Vol, topology_list=mode_list)

### 6 - 2: plot CV vs. Multiverse plots in SR

#### 6 - 2 - a: Signal

In [ ]:
plot_univ_hists(sr_signal_univ_events, sr_signal_sel_reco_cv, "GENIE", varcfg_p_mu, categ_name="Signal")

#### 6 - 2 - b: backgrounds

In [ ]:
for i in range(len(mode_list))[1:]:
    this_mode_name = mode_labels[i]
    this_categ_univ_events = sr_bkg_univ_events[:, i - 1, :]
    plot_univ_hists(this_categ_univ_events, sr_bkg_sec_rec_cv[i - 1], "GENIE", varcfg_p_mu, categ_name=this_mode_name)

### 6 - 2: plot CV vs. Multiverse plots in SB

#### 6 - 2 - a: Signal

In [ ]:
plot_univ_hists(sb_signal_univ_events, sb_signal_sel_reco_cv, "GENIE", varcfg_p_mu, categ_name="Signal")

#### 6 - 2 - b: Background

In [ ]:
for i in range(len(mode_list))[1:]:
    this_mode_name = mode_labels[i]
    this_categ_univ_events = sr_bkg_univ_events[:, i - 1, :]
    plot_univ_hists(this_categ_univ_events, sr_bkg_sec_rec_cv[i - 1], "GENIE", varcfg_p_mu, categ_name=this_mode_name)

## 7. Covariance Matrices

For CCBC, we consider signal events (m) and background events (b) in signal region (s) and sideband (c)

We need these self-covariance matrices
1. Cov(m_s, m_s): cov_ms_ms
2. Cov(b_s, b_s): cov_bs_bs
3. Cov(m_c, m_c): cov_mc_mc
4. Cov(b_c, b_c): cov_bc_bc

In addition, we also need

5. Cov(m_s, b_s), Cov(b_s, m_s): cov_ms_bs, cov_bs_ms
6. Cov(m_s, m_c), Cov(m_c, m_s): cov_ms_mc, cov_mc_ms
7. Cov(m_s, b_c), Cov(b_c, m_s): cov_ms_bc, cov_bc_ms
8. Cov(b_s, m_c), Cov(m_c, b_s): cov_bs_mc, cov_mc_bs
9. Cov(b_s, b_c), Cov(b_c, b_s): cov_bs_bc, cov_bc_bs
10. Cov(m_c, b_c), Cov(b_c, m_c): cov_mc_bc, cov_bc_mc

### 7 - 1: Signal region, cov_ms_ms and cov_bs_bs

In [ ]:
cov_ms_ms = get_covariance_matrix_self(sr_signal_univ_events, sr_signal_sel_reco_cv)
plot_heatmap(cov_ms_ms["cov"], varcfg_p_mu, title="GENIE syst. Cov. Signal in SR")

In [ ]:
### For the total background
sr_total_bkg_sec_rec_cv = sr_bkg_sec_rec_cv.sum(axis=0)
sr_total_bkg_univ_events = sr_bkg_univ_events.sum(axis=1)
cov_bs_bs = get_covariance_matrix_self(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)
plot_heatmap(cov_bs_bs["cov"], varcfg_p_mu, title="GENIE syst. Cov. Background in SR")

### 7 - 2: Sideband, cov_mc_mc and cov_bc_bc

In [ ]:
cov_mc_mc = get_covariance_matrix_self(sb_signal_univ_events, sb_signal_sel_reco_cv)
plot_heatmap(cov_mc_mc["cov"], varcfg_p_mu, title="GENIE syst. Cov. Signal in SB")

In [ ]:
### For the total background
sb_total_bkg_sec_rec_cv = sb_bkg_sec_rec_cv.sum(axis=0)
sb_total_bkg_univ_events = sb_bkg_univ_events.sum(axis=1)
cov_bc_bc = get_covariance_matrix_self(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
plot_heatmap(cov_bc_bc["cov"], varcfg_p_mu, title="GENIE syst. Cov. Background in SB")

### 7 - 3: Crossing covariance matrices

In [ ]:
cov_ms_bs = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)
cov_bs_ms = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_ms_mc = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)
cov_mc_ms = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_ms_bc = get_covariance_matrix(sr_signal_univ_events, sr_signal_sel_reco_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_ms = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sr_signal_univ_events, sr_signal_sel_reco_cv)

cov_bs_mc = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)
cov_mc_bs = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)

cov_bs_bc = get_covariance_matrix(sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_bs = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sr_total_bkg_univ_events, sr_total_bkg_sec_rec_cv)

cov_mc_bc = get_covariance_matrix(sb_signal_univ_events, sb_signal_sel_reco_cv, sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv)
cov_bc_mc = get_covariance_matrix(sb_total_bkg_univ_events, sb_total_bkg_sec_rec_cv, sb_signal_univ_events, sb_signal_sel_reco_cv)

## 8. Save CV hists and Cov matrices

In [ ]:
ret_dict = {
    "ms": sr_signal_sel_reco_cv,
    "bs": sr_total_bkg_sec_rec_cv,
    "mc": sb_signal_sel_reco_cv,
    "bc": sb_total_bkg_sec_rec_cv,
    "cov_ms_ms": cov_ms_ms,
    "cov_bs_bs": cov_bs_bs,
    "cov_mc_mc": cov_mc_mc,
    "cov_bc_bc": cov_bc_bc,
    "cov_ms_bs": cov_ms_bs,
    "cov_bs_ms": cov_bs_ms,
    "cov_ms_mc": cov_ms_mc,
    "cov_mc_ms": cov_mc_ms,
    "cov_ms_bc": cov_ms_bc,
    "cov_bc_ms": cov_bc_ms,
    "cov_bs_mc": cov_bs_mc,
    "cov_mc_bs": cov_mc_bs,
    "cov_bs_bc": cov_bs_bc,
    "cov_bc_bs": cov_bc_bs,
    "cov_mc_bc": cov_mc_bc,
    "cov_bc_mc": cov_bc_mc,
    "response": response,
    "true_signal": sighist_res["nevts_allmc"],
}

np.savez(output_str + "_syst_cov_matrices.npz", **ret_dict)